# Statistical Modelling Project II: Planning a Term Under Uncertainty

Project I asked what drives exam scores. This project asks a different
question given that model: **how should a student spend a limited term to
get the best result?**

It reuses Project I's fitted coefficients and dataset wherever possible, and
clearly flags every additional number as an assumption made for this project.
The same underlying scenario is modelled at increasing levels of realism:

1. **Scenario**: parameters and what each one is based on.
2. **LP baseline**: the best plan if the world were deterministic and linear.
3. **MDP**: the exact optimal policy once outcomes are uncertain and
   decisions are sequential.
4. **MCTS**: approximate planning by simulation, validated against the MDP's
   exact answer.
5. **Risk-sensitive extension**: maximising the chance of reaching a target
   grade, rather than the average score.
6. **Results**: a side-by-side comparison of all four approaches.
7. **Recommendations**: what the plan means for the student.

Random seed fixed at 42 throughout, so every number here is reproducible.

## 1. Scenario

Every parameter used from here on falls into one of two categories, and both
are kept visibly separate throughout this notebook and in `code/common.py`:

- **From Project I**: a real, fitted number (or a real descriptive statistic
  of Project I's dataset), taken as given.
- **Assumption for this project**: a choice made to turn Project I's
  regression into a concrete planning scenario. Project I never modelled
  decisions or time, so these numbers do not exist anywhere until this
  project defines them.

The one genuinely new descriptive statistic needed at this stage is the mean
weekly study hours across Project I's 320 students. Project I's Group study
and Tutoring effects are a fixed points-per-week bonus for using that method
at all, not a per-hour rate, so Stage 1's hours-allocation linear programme
needs them converted to a per-hour premium. That conversion divides each
dummy coefficient by this mean, which is why it is computed here rather than
simply assumed.

In [1]:
import sys
sys.path.append("../code")

import pandas as pd
import common

df = common.load_student_data()
print(f"Students in Project I's dataset: {len(df)}")
print(f"Mean weekly study hours: {common.MEAN_WEEKLY_STUDY_HOURS}")
df.head()

Students in Project I's dataset: 320
Mean weekly study hours: 11.9094


,student_id,study_hours_per_week,attendance_rate,sleep_hours,prior_gpa,extracurricular_hours,study_method,exam_score
0,S0001,13.28,66.5,7.48,3.38,1,Group study,53.2
1,S0002,6.53,77.5,6.68,2.79,3,Self-study,58.6
2,S0003,15.69,92.8,6.81,3.62,5,Self-study,92.0
3,S0004,8.39,82.9,5.91,3.18,4,Self-study,67.3
4,S0005,11.22,70.1,8.14,2.94,3,Self-study,56.9


### Parameter table: from Project I vs assumption for this project

The table below is generated directly from `code/common.py`, so it can never
drift out of sync with the values the rest of this notebook actually uses.

In [2]:
rows = [
    ("Mean weekly study hours", common.MEAN_WEEKLY_STUDY_HOURS, "From Project I (dataset mean)"),
    ("Study hours coefficient (points per hour)", common.PROJECT_I_STUDY_HOURS_COEF, "From Project I (fitted)"),
    ("Attendance coefficient (points per pp)", common.PROJECT_I_ATTENDANCE_COEF, "From Project I (fitted)"),
    ("Prior GPA coefficient (points per GPA point)", common.PROJECT_I_PRIOR_GPA_COEF, "From Project I (fitted)"),
    ("Extracurricular coefficient (points per hour)", common.PROJECT_I_EXTRACURRICULAR_COEF, "From Project I (fitted)"),
    ("Group study dummy (points per week)", common.PROJECT_I_GROUP_STUDY_DUMMY, "From Project I (fitted)"),
    ("Tutoring dummy (points per week)", common.PROJECT_I_TUTORING_DUMMY, "From Project I (fitted)"),
    ("Sleep coefficient, estimated (not significant)", common.PROJECT_I_SLEEP_COEF_ESTIMATED, "From Project I (fitted)"),
    ("Sleep penalty, true generating value", common.TRUE_SLEEP_PENALTY_PER_HOUR_BELOW_6, "From Project I (true, generating)"),
    ("Exam score noise SD", common.TRUE_NOISE_SD, "From Project I (true, generating)"),
    ("Group study premium per hour", round(common.LP_GROUP_STUDY_PREMIUM_PER_HOUR, 4), "Assumption (derived: dummy / mean hours)"),
    ("Tutoring premium per hour", round(common.LP_TUTORING_PREMIUM_PER_HOUR, 4), "Assumption (derived: dummy / mean hours)"),
    ("Weekly hours budget", common.LP_TOTAL_HOURS_BUDGET, "Assumption"),
    ("Tutoring hours cap", common.LP_TUTORING_MAX_HOURS, "Assumption"),
    ("Group study hours cap", common.LP_GROUP_STUDY_MAX_HOURS, "Assumption"),
    ("Scheduled lecture hours", common.LP_SCHEDULED_LECTURE_HOURS, "Assumption"),
    ("Extracurricular minimum hours", common.LP_EXTRACURRICULAR_MIN_HOURS, "Assumption"),
    ("Term length (weeks)", common.MDP_TERM_WEEKS, "Assumption"),
    ("Knowledge scale (0 to k_max)", common.MDP_KNOWLEDGE_MAX, "Assumption"),
    ("Tutoring sessions available", common.MDP_TUTORING_SESSIONS_MAX, "Assumption"),
    ("Starting state (t, k, f, s)", tuple(common.MDP_START_STATE.values()), "Assumption"),
    ("P(self-study gains +1 knowledge)", common.MDP_P_SELF_STUDY, "Assumption (baseline)"),
    ("P(group study gains +1 knowledge)", common.MDP_P_GROUP_STUDY, "Assumption (ratio anchored to Project I)"),
    ("P(tutoring gains +1 / +2 knowledge)", (common.MDP_P_TUTORING_PLUS1, common.MDP_P_TUTORING_PLUS2), "Assumption (ratio anchored to Project I)"),
    ("Fatigue success multipliers (fresh, tired, exhausted)", tuple(common.MDP_FATIGUE_FACTOR.values()), "Assumption"),
    ("P(studying raises fatigue by 1)", common.MDP_P_FATIGUE_INCREASE, "Assumption"),
    ("Terminal score at k = 0 / k = 10", (common.terminal_expected_score(0), common.terminal_expected_score(10)), "Assumption (calibrated to Project I's score range)"),
    ("Risk-sensitive target grade", common.RISK_TARGET_GRADE, "Assumption (example: a scholarship threshold)"),
]
param_table = pd.DataFrame(rows, columns=["Parameter", "Value", "Source"])
param_table

,Parameter,Value,Source
0,Mean weekly study hours,11.9094,From Project I (dataset mean)
1,Study hours coefficient (points per hour),1.13,From Project I (fitted)
2,Attendance coefficient (points per pp),0.28,From Project I (fitted)
3,Prior GPA coefficient (points per GPA point),7.73,From Project I (fitted)
4,Extracurricular coefficient (points per hour),-0.52,From Project I (fitted)
5,Group study dummy (points per week),1.83,From Project I (fitted)
6,Tutoring dummy (points per week),5.56,From Project I (fitted)
7,"Sleep coefficient, estimated (not significant)",0.4,From Project I (fitted)
8,"Sleep penalty, true generating value",-1.8,"From Project I (true, generating)"
9,Exam score noise SD,7.5,"From Project I (true, generating)"


## 2. LP baseline

*To be built in Phase 2.*

## 3. MDP

*To be built in Phase 3.*

## 4. MCTS

*To be built in Phase 4.*

## 5. Risk-sensitive extension

*To be built in Phase 5.*

## 6. Results

*To be built in Phase 6.*

## 7. Recommendations

*To be built in Phase 6.*